# Model Fine-tuning

Training and fine-tuning models on hotel-specific data.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

In [3]:
import os

# Disable torch.compile / torchdynamo to avoid pickling ConfigModuleInstance
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCH_COMPILE"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

In [4]:
import sys
sys.path.append("/content/drive/MyDrive/HotelPromenade-AI-Intelligence/src")

In [5]:
from finetuning.trainer import *

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [6]:
JSONL_PATH = "/content/drive/MyDrive/HotelPromenade-AI-Intelligence/data/finetune/hotel_finetune.jsonl"

  **Load base model**

In [7]:

model, tokenizer = load_base_model(
    model_name="unsloth/llama-3.1-8b-instruct-bnb-4bit",
    max_seq_length= 800 ,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

**Format datasets**

In [8]:
full_ds = format_dataset_with_template(
    JSONL_PATH,
    tokenizer
)
dataset_split = full_ds.train_test_split(
    test_size=0.1,   # 10% eval
    seed=42
)

train_ds = dataset_split["train"]
eval_ds  = dataset_split["test"]


Generating train split: 0 examples [00:00, ? examples/s]

Applying chat template to dataset (num_proc=1):   0%|          | 0/49 [00:00<?, ? examples/s]

In [9]:
def get_length_stats(dataset, tokenizer):
    lengths = []

    for i in range(len(dataset)):
        ids = tokenizer(
            dataset[i]["text"],
            add_special_tokens=False
        )["input_ids"]
        lengths.append(len(ids))

    max_len = max(lengths)
    min_len = min(lengths)
    avg_len = sum(lengths) / len(lengths)

    print("Min length:", min_len)
    print("Max length:", max_len)
    print("Avg length:", round(avg_len, 2))

    return lengths

train_lengths = get_length_stats(train_ds, tokenizer)

Min length: 409
Max length: 786
Avg length: 574.48


**Apply LoRA**

In [12]:

model = apply_lora(
    model,
    r=8,
    lora_alpha= 16,
    target_modules=[
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

Unsloth 2026.3.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 20,971,520 || all params: 8,051,232,768 || trainable%: 0.2605


**Create trainer**

In [13]:
trainer = create_trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    output_dir="./results-faq-lora",
    num_train_epochs= 1 ,
    learning_rate=1e-4,
    per_device_train_batch_size= 1,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    logging_steps=5,
    save_strategy="epoch",
    seed=42,
    max_seq_length=800,
    packing=False,
)


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/44 [00:00<?, ? examples/s]

num_proc must be <= 5. Reducing num_proc to 5 for dataset of size 5.


Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/5 [00:00<?, ? examples/s]

**Train & Save the model**

In [ ]:

result = train_model(trainer)
#  Save on drive
save_model(trainer, tokenizer, "/content/drive/MyDrive/results-faq-lora")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 44 | Num Epochs = 1 | Total steps = 11
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 20,971,520 of 8,051,232,768 (0.26% trained)


Step,Training Loss
5,2.222000
10,1.821400


Final training loss: 1.9798
